# Splits! Demo

This notebook demonstrates how to load a demographic-demographic-topic Pyserini index, query it with a predefined lexicon, and compute the Lift and Triviality metrics for a target demographic

In [ ]:
# 1. Import Core Logic Elements
from core_logic import (
    query_bm25_index, 
    lift_at_k, 
    lift_ci,
    compute_keyword_similarity
)
import json

# Lift Metric

Here we load the desired index, pick the target demographic, define the PSLP lexicon, and finally calculate the lift!

In [ ]:
# 2. Execution Example

# Configuration
index_name = 'Video_Games_-General_Discussion-_black-jewish' # see lexica/bm25_indexes/ for available indexes
index_path = f'lexica/bm25_indexes/{index_name}'
target_demo = 'jewish'

# Define the Lexica in a dictionary for easy iteration
lexica = {
    "Non-Lifting": [
        "the", "be", "to", "of", "and",
        "a", "in", "that", "have", "I",
        "it", "for", "not", "on", "with",
        "he", "as", "you", "do", "at"
    ],
    "Trivial": [
        "Torah", "I am Jewish", "Shabbat", "kosher", "synagogue",
        "Hanukkah", "Passover", "Yom Kippur", "Rosh Hashanah", "Bar Mitzvah"
    ]
}

# Define the thresholds (k values)
thresholds = [0.005, 0.01, 0.02, 0.05, 0.10]

# Run the loop
for label, keywords in lexica.items():
    print(f"\n{'='*20} Evaluating Lexicon: {label} {'='*20}")
    
    # Run search for the current lexicon
    df_results = query_bm25_index(index_path, keywords, doc_count=2000)
    print(f"Total retrieved docs: {len(df_results)}")

    # Compute and print lift for each threshold
    for k in thresholds:
        lift_val = lift_at_k(df_results, target_demo, k=k)
        pval, ci_lower, ci_upper = lift_ci(df_results, target_demo, k=k)
        
        # Display results with dynamic percentage formatting
        k_pct = f"{k*100:.1f}%" if k < 0.01 else f"{int(k*100)}%"
        print(f"Lift@{k_pct:<5}: {lift_val:.3f} (p-value: {pval:.6f}, 95% CI: [{ci_lower:.3f}, {ci_upper:.3f}])")

    # Optional: Display top results for this specific lexicon
    # print(df_results.sort_values(by='score', ascending=False).head(5))

# 4. Triviality Metric

Calculate Triviality between the demographic seed set and the lexicon using `SubspaceBERTScore`.

In [ ]:
# 3. Execution Example - Triviality Score

# load the seed lexica
with open('lexica/demo_seed_words.json', 'r') as f:
    seed_lexica = json.load(f)

# Example usage comparing triviality of trivial lexicon and non-lifting lexicon:
target_seed_words = seed_lexica[target_demo]

# Non-Lifting Lexicon Similarity
similarity_metrics = compute_keyword_similarity(target_seed_words, lexica["Non-Lifting"])
print("\nTriviality Score for Non-Lifting Lexicon:")
print(f"{similarity_metrics['Recall']:.4f}")

# Trivial Lexicon Similarity
similarity_metrics = compute_keyword_similarity(target_seed_words, lexica["Trivial"])
print("\nTriviality Score for Trivial Lexicon:")
print(f"{similarity_metrics['Recall']:.4f}")